# finance / portfolio / mix

Expected return and volatility of an investment mix of risky and defensive assets.

| Term | In This Notebook |
|---|---|
| **Intent** | Give the two numbers every mix decision starts from, expected return and volatility, for any share held in risky assets. Placeholder: the risky and defensive assets and their correlation are recovered from synthetic points |
| **Project Flavour** | `pyfunc`, served by `mlserver` (`model.md` 3.2) |
| **Data Splits** | `training` ten risky shares with the mix return and volatility; `validation` five further; `backtest` ten across the range, held back for acceptance; `golden` the exact shares that become `golden.json` |

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** put `workbench` on the path, load `env`, and declare the model name, dataset names and signature names as constants (`model.md` 6.6).

In [ ]:
import json
from os.path import join

from workbench.config import NOTEBOOK_DIR
from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import provenance, record_project, scratch_project_dir

env = init_env()

WORKSPACE = "finance/portfolio/mix"
MODEL_NAME = "finance_portfolio_mix"
TARGETS = [{"kind": "served"}]
TRAINING = "portfolio_mix_training_2026-11-05_2026-11-05"
VALIDATION = "portfolio_mix_validation_2026-11-05_2026-11-05"
BACKTEST = "portfolio_mix_backtest_2026-11-05_2026-11-05"
GOLDEN = "portfolio_mix_golden"
INPUTS = ["finance_portfolio_mix_risky_share_pct"]
OUTPUTS = ["finance_portfolio_mix_return_pct", "finance_portfolio_mix_volatility_pct"]

## Capture Data

- **Stage:** 1, capture data (`model.md` 7.1).
- **Intent:** write the `training`, `validation` and `backtest` files only when they are absent (`model.md` 6.5). The source is synthetic, risky assets at 6 percent return and 15 percent volatility, defensive at 2 and 5, correlation 0.2, so no query runs.

In [ ]:
from math import sqrt


def mix(share):
    weight = share / 100.0
    variance = (weight * 15.0) ** 2 + ((1.0 - weight) * 5.0) ** 2 + 2.0 * 0.2 * weight * (1.0 - weight) * 15.0 * 5.0
    return 2.0 + 4.0 * weight, sqrt(variance)


def labelled(shares):
    return {INPUTS[0]: shares, OUTPUTS[0]: [round(mix(share)[0], 6) for share in shares], OUTPUTS[1]: [round(mix(share)[1], 6) for share in shares]}


training_pl, training_path = ensure_data(WORKSPACE, "training", TRAINING, lambda: labelled([0.0, 10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0, 90.0]))
validation_pl, validation_path = ensure_data(WORKSPACE, "validation", VALIDATION, lambda: labelled([5.0, 25.0, 45.0, 65.0, 85.0]))
backtest_pl, backtest_path = ensure_data(WORKSPACE, "backtest", BACKTEST, lambda: labelled([15.0, 35.0, 55.0, 75.0, 95.0, 100.0, 0.0, 50.0, 60.0, 90.0]))
print("training", training_pl.height, "validation", validation_pl.height, "backtest", backtest_pl.height)

## Fit

- **Stage:** 3, fit.
- **Intent:** fit the return as a line and the variance as a quadratic in the risky share on `training` only (R6), which is exactly the two asset blend, so the two assets and their correlation are recoverable.

In [ ]:
import numpy as np

shares = training_pl[INPUTS[0]].to_numpy() / 100.0
return_slope, return_intercept = np.polyfit(shares, training_pl[OUTPUTS[0]].to_numpy(), 1)
variance_coefficients = np.polyfit(shares, training_pl[OUTPUTS[1]].to_numpy() ** 2, 2)
PARAMS = {
    "return_intercept": round(float(return_intercept), 4),
    "return_slope": round(float(return_slope), 4),
    "variance_square": round(float(variance_coefficients[0]), 4),
    "variance_linear": round(float(variance_coefficients[1]), 4),
    "variance_constant": round(float(variance_coefficients[2]), 4),
}
print("fitted", PARAMS)
assert PARAMS["return_intercept"] == 2.0 and PARAMS["return_slope"] == 4.0, "the fit did not recover the asset returns"

## Evaluate

- **Stage:** 4, evaluate.
- **Intent:** write `model.py` from the fitted values, build the signature, run it through the library runtime and check the `validation` and `backtest` error against the limit the notebook owns (R4, R14).

In [ ]:
from workbench.model import build_signature, evaluate_acceptance, measure, signature_input, signature_output
from workbench.replay import model_predict

MODEL_TEMPLATE = """from math import sqrt

import pandas as pd
from mlflow.models import set_model
from mlflow.pyfunc import PythonModel

INPUT = "finance_portfolio_mix_risky_share_pct"
OUTPUT_RETURN = "finance_portfolio_mix_return_pct"
OUTPUT_VOLATILITY = "finance_portfolio_mix_volatility_pct"
RETURN_INTERCEPT = {return_intercept!r}
RETURN_SLOPE = {return_slope!r}
VARIANCE_SQUARE = {variance_square!r}
VARIANCE_LINEAR = {variance_linear!r}
VARIANCE_CONSTANT = {variance_constant!r}


class Mix(PythonModel):
    def predict(self, context, model_input: pd.DataFrame, params=None):
        weights = [share / 100.0 for share in model_input[INPUT]]
        return pd.DataFrame(
            {{
                OUTPUT_RETURN: [RETURN_INTERCEPT + RETURN_SLOPE * weight for weight in weights],
                OUTPUT_VOLATILITY: [sqrt(max(VARIANCE_SQUARE * weight**2 + VARIANCE_LINEAR * weight + VARIANCE_CONSTANT, 0.0)) for weight in weights],
            }}
        )


set_model(Mix())
"""

model_path = join(NOTEBOOK_DIR, WORKSPACE, "model.py")
with open(model_path, "w") as model_file:
    model_file.write(MODEL_TEMPLATE.format(**PARAMS))

signature = build_signature(
    MODEL_NAME,
    [
        signature_input("finance_portfolio_mix_risky_share_pct", "percent", "Share of the portfolio held in risky assets, 0 to 100 percent."),
    ],
    [
        signature_output("finance_portfolio_mix_return_pct", "percent", "Expected annual return of the mix, percent.", datatype="FP64"),
        signature_output("finance_portfolio_mix_volatility_pct", "percent", "Annualised volatility of the mix, percent.", datatype="FP64"),
    ],
)
predict = model_predict(model_path, "pyfunc", signature)

In [ ]:
LIMIT = 0.05
SPLITS = (("validation", validation_pl), ("backtest", backtest_pl))


def acceptance_checks():
    answers = {split: predict({INPUTS[0]: frame[INPUTS[0]].to_list()}) for split, frame in SPLITS}
    return [
        {"split": split, "output": output, "name": "rmse", "value": measure(frame[output].to_list(), answers[split][output])["rmse"], "limit": LIMIT, "direction": "max"}
        for split, frame in SPLITS
        for output in OUTPUTS
    ]


evaluation = evaluate_acceptance(acceptance_checks, {"validation": [validation_path], "backtest": [backtest_path]})
for entry in evaluation["checks"]:
    print(entry["split"], entry["output"], entry["name"], "measured", entry["value"], "passed", entry["passed"])
print("acceptance", "passed" if evaluation["passed"] else "failed, the shipped model is outside the limit")

## Package

- **Stage:** 5, package.
- **Intent:** write the golden file from the shipped runtime, copy `model.py` and the evaluation into a project, verify it and replay the golden through the library runtime (R1, R3, R14).

In [ ]:
def golden_samples():
    xs = [0.0, 25.0, 50.0, 75.0, 100.0]
    answers = predict({INPUTS[0]: xs})
    return {INPUTS[0]: [*xs, None], **{output: [*answers[output], None] for output in OUTPUTS}, "expect": ["ok"] * len(xs) + ["error"]}

In [ ]:
from workbench.model import package_project, register_preview
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, golden_samples)

project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "pyfunc",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic two asset blend, training split",
        [training_path, golden_path],
        "line for return and quadratic for variance",
        implementation="mlflow.pyfunc model.py, requires mlflow, pandas and the standard library",
    ),
    model_path=model_path,
    evaluation=evaluation,
    sequence="from all defensive to all risky",
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGETS)
print("replayed the golden sequences exactly through the shipped runtime")

## Register

- **Stage:** 6, register.
- **Intent:** register the verified project with the alias `preview`. A person sets `release` after review and the parity gate (R5).

In [ ]:
version = register_preview(project_dir, MODEL_NAME, env["MLFLOW_TRACKING_URI"], TARGETS, metadata["snapshot_version"])
print("registered version", version)